In [1]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

--2026-10-09 19:01:39--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.02s   

2026-10-09 19:01:39 (16.3 MB/s) - ‘course_lead_scoring_2026.csv’ saved [278746/278746]



In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('course_lead_scoring_2026.csv')

In [5]:
numerical_cols = df.select_dtypes(include=['number']).columns
categorical_cols = df.select_dtypes(exclude=['number']).columns

In [6]:
df[categorical_cols] = df[categorical_cols].fillna('NA')
df[numerical_cols] = df[numerical_cols].fillna(0.0)

In [7]:
print("Mode:", df['industry'].mode()[0])

Mode: technology


In [8]:
corr_matrix = df[numerical_cols].corr()

pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

print("\nWartości korelacji dla podanych par:")
for col1, col2 in pairs:
    val = corr_matrix.loc[col1, col2]
    print(f"{col1} <-> {col2}: {val:.4f}")


Wartości korelacji dla podanych par:
interaction_count <-> lead_score: 0.9157
number_of_courses_viewed <-> lead_score: 0.7572
number_of_courses_viewed <-> interaction_count: 0.7216
annual_income <-> interaction_count: 0.1228


In [9]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

Train: 3000, Val: 1000, Test: 1000


In [10]:
from sklearn.metrics import mutual_info_score

cat_features = ['industry', 'location', 'lead_source', 'employment_status']

def calculate_mi(series):
    return mutual_info_score(series, y_train)

# Zastosowanie do zbioru treningowego i zaokrąglenie do 2 miejsc po przecinku
mi_scores = df_train[cat_features].apply(calculate_mi)
mi_scores = mi_scores.round(2).sort_values(ascending=False)

print(mi_scores)

lead_source          0.03
employment_status    0.02
location             0.00
industry             0.00
dtype: float64


In [11]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

acc = accuracy_score(y_val, y_pred)
print(f"Pełna dokładność: {acc}")
print(f"Odpowiedź (round(acc, 2)): {round(acc, 2)}")

Pełna dokładność: 0.645
Odpowiedź (round(acc, 2)): 0.65


In [12]:
orig_score = acc  # zmienna acc z poprzedniego kroku

features = list(df_train.columns)
diffs = {}

for feature in features:
    subset = [f for f in features if f != feature]
    
    train_dicts = df_train[subset].to_dict(orient='records')
    val_dicts = df_val[subset].to_dict(orient='records')
    
    dv = DictVectorizer(sparse=False)
    X_train_sub = dv.fit_transform(train_dicts)
    X_val_sub = dv.transform(val_dicts)
    
    model = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )
    model.fit(X_train_sub, y_train)
    
    y_pred = model.predict(X_val_sub)
    score_without = accuracy_score(y_val, y_pred)
    
    difference = orig_score - score_without
    diffs[feature] = {
        'score_without': score_without,
        'difference': difference
    }
diff_df = pd.DataFrame(diffs).T.sort_values(by='difference')
print("Różnice w accuracy (original - without feature):")
display(diff_df)

Różnice w accuracy (original - without feature):


,score_without,difference
annual_income,0.724,-0.079
industry,0.645,0.000
location,0.645,0.000
employment_status,0.645,0.000
lead_score,0.644,0.001
number_of_courses_viewed,0.643,0.002
lead_source,0.642,0.003
interaction_count,0.601,0.044


In [13]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

results = []

for c in C_values:

    model = LogisticRegression(
        solver='liblinear',
        C=c,
        max_iter=1000,
        random_state=42
    )
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    score = accuracy_score(y_val, y_pred)
    
    results.append({
        'C': c,
        'accuracy_raw': score,
        'accuracy_rounded': round(score, 3)
    })

results_df = pd.DataFrame(results)
print(results_df)

          C  accuracy_raw  accuracy_rounded
0  0.000001         0.598             0.598
1  0.000010         0.598             0.598
2  0.000100         0.613             0.613
3  0.001000         0.645             0.645
